In [1]:
import pandas as pd
print("version",pd.__version__)

version 3.0.5


# Customer Churn Prediction - Data Preprocessing

## Objective
Prepare customer data for a machine learning model that predicts customer churn.

## Target
- `churn = 0` → Customer stays
- `churn = 1` → Customer leaves

In [5]:
import pandas as pd
import numpy as np

# Make the results reproducible
np.random.seed(42)

# Number of customers
n = 1000

# Create customer IDs
customer_id = [f"C{i:04d}" for i in range(1, n + 1)]

# Generate customer features
age = np.random.randint(18, 70, n)

tenure_months = np.random.randint(1, 61, n)

monthly_charges = np.round(
    np.random.uniform(30, 120, n), 2
)

support_calls = np.random.randint(0, 10, n)

contract_type = np.random.choice(
    ["Monthly", "Yearly", "Two-Year"],
    n,
    p=[0.55, 0.30, 0.15]
)

usage_hours = np.round(
    np.random.uniform(10, 200, n), 1
)

# Create a probability of churn
churn_score = (
    0.03
    + 0.015 * support_calls
    + 0.004 * monthly_charges
    - 0.012 * tenure_months
    + 0.10 * (contract_type == "Monthly")
)

# Convert score into probability
churn_probability = 1 / (1 + np.exp(-churn_score))

# Generate churn labels
churn = np.random.binomial(
    1,
    np.clip(churn_probability, 0, 1),
    n
)

# Create DataFrame
df = pd.DataFrame({
    "customer_id": customer_id,
    "age": age,
    "tenure_months": tenure_months,
    "monthly_charges": monthly_charges,
    "support_calls": support_calls,
    "contract_type": contract_type,
    "usage_hours": usage_hours,
    "churn": churn
})

df.head()

,customer_id,age,tenure_months,monthly_charges,support_calls,contract_type,usage_hours,churn
0,C0001,56,35,103.62,2,Monthly,19.6,1
1,C0002,69,51,104.89,0,Yearly,17.5,1
2,C0003,46,15,75.67,3,Monthly,119.1,1
3,C0004,32,25,30.57,1,Monthly,84.7,0
4,C0005,60,55,55.83,1,Monthly,15.4,0


In [7]:
from pathlib import Path
raw_source=Path("../data/raw/churn.csv")
df.to_csv(raw_source,index=False)
print(f"raw data saved to",{raw_source})

raw data saved to {WindowsPath('../data/raw/churn.csv')}


## 1. Data Inspection

Before preprocessing, we inspect:

- Dataset dimensions
- Column names
- Data types
- Missing values
- Duplicate records
- Statistical distribution
- Target distribution

In [8]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 1000
Columns: 8


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      1000 non-null   str    
 1   age              1000 non-null   int32  
 2   tenure_months    1000 non-null   int32  
 3   monthly_charges  1000 non-null   float64
 4   support_calls    1000 non-null   int32  
 5   contract_type    1000 non-null   str    
 6   usage_hours      1000 non-null   float64
 7   churn            1000 non-null   int32  
dtypes: float64(2), int32(4), str(2)
memory usage: 47.0 KB


In [10]:
df.describe()

,age,tenure_months,monthly_charges,support_calls,usage_hours,churn
count,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,43.81900,31.076000,75.163240,4.434000,103.146300,0.517000
std,14.99103,17.207391,26.044527,2.839314,54.519635,0.499961
min,18.00000,1.000000,30.290000,0.000000,10.000000,0.000000
25%,31.00000,17.000000,51.877500,2.000000,55.775000,0.000000
50%,44.00000,32.000000,75.975000,4.000000,101.900000,1.000000
75%,56.00000,46.000000,97.410000,7.000000,150.200000,1.000000
max,69.00000,60.000000,119.950000,9.000000,199.600000,1.000000


In [11]:
df["churn"].value_counts()

churn
1    517
0    483
Name: count, dtype: int64

In [12]:
df["contract_type"].value_counts()

contract_type
Monthly     547
Yearly      319
Two-Year    134
Name: count, dtype: int64

## Feature Selection

We will use customer behavioral and account information to predict churn.

`customer_id` is an identifier and will not be used as a model feature.

Target:
- churn

In [13]:
FEATURES = [
    "age",
    "tenure_months",
    "monthly_charges",
    "support_calls",
    "contract_type",
    "usage_hours"
]

TARGET = "churn"

print("Features:", FEATURES)
print("Target:", TARGET)

Features: ['age', 'tenure_months', 'monthly_charges', 'support_calls', 'contract_type', 'usage_hours']
Target: churn


In [14]:
df.shape

(1000, 8)

In [15]:
df.isnull().sum()

customer_id        0
age                0
tenure_months      0
monthly_charges    0
support_calls      0
contract_type      0
usage_hours        0
churn              0
dtype: int64

In [16]:
df_raw = df.copy()
df_raw.loc[10, "age"] = np.nan
df_raw.loc[25, "monthly_charges"] = np.nan
df_raw.loc[50, "contract_type"] = np.nan
df_raw.isnull().sum()

customer_id        0
age                1
tenure_months      0
monthly_charges    1
support_calls      0
contract_type      1
usage_hours        0
churn              0
dtype: int64

In [18]:
df_raw = pd.concat(
    [df_raw, df_raw.iloc[[100]]],
    ignore_index=True
)
df_raw.duplicated().sum()

np.int64(2)

In [19]:
df_raw.loc[70, "monthly_charges"] = 999999

In [22]:
df_raw['monthly_charges'].max()

np.float64(999999.0)

In [23]:
from pathlib import Path

raw_path = Path("../data/raw/churn.csv")

df_raw.to_csv(raw_path, index=False)

print(f"Raw dataset saved: {raw_path}")

Raw dataset saved: ..\data\raw\churn.csv


In [24]:
print("Shape:", df_raw.shape)

print("\nMissing values:")
print(df_raw.isnull().sum())

print("\nDuplicate rows:")
print(df_raw.duplicated().sum())

print("\nData types:")
print(df_raw.dtypes)

Shape: (1002, 8)

Missing values:
customer_id        0
age                1
tenure_months      0
monthly_charges    1
support_calls      0
contract_type      1
usage_hours        0
churn              0
dtype: int64

Duplicate rows:
2

Data types:
customer_id            str
age                float64
tenure_months        int32
monthly_charges    float64
support_calls        int32
contract_type          str
usage_hours        float64
churn                int32
dtype: object
